# Dual-Target Grounded-Execute ChartQA — Kaggle runner

Ten lines. Everything real lives in the installed package, so this notebook never
needs editing when the code changes — only the final command does.

**Before running:** add these under *Add-ons → Secrets*
`GITHUB_TOKEN`, `GITHUB_USER`, `HF_TOKEN`, and optionally `WANDB_API_KEY`.

**Settings:** Accelerator = GPU T4 x2 (only one is used), Internet = On, Persistence = Files.


In [ ]:
from kaggle_secrets import UserSecretsClient
import os, subprocess, sys

s = UserSecretsClient()
for k in ("GITHUB_TOKEN", "GITHUB_USER", "HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[k] = s.get_secret(k)
    except Exception:
        print(f"secret {k} not set (continuing)")


In [ ]:
REPO = f"https://{os.environ['GITHUB_TOKEN']}@github.com/{os.environ['GITHUB_USER']}/chartqa-dual-target.git"
BRANCH = "main"
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "/kaggle/working/repo"], check=True)


In [ ]:
# -q keeps the log readable; --no-deps is NOT used because the pins matter.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/kaggle/working/repo[gpu,eval]"], check=True)


In [ ]:
# Confirm the environment module resolved Kaggle paths and found the GPU.
subprocess.run([sys.executable, "-c",
                "from chartqa_dt.env import get_env; print(get_env().describe())"], check=True)


In [ ]:
# ---- the actual job -------------------------------------------------------
# Phase 2 smoke test:
#   cdt-train --stage smoke --config configs/model_qwen3vl2b.yaml
# Zero-shot eval:
#   cdt-eval --dataset chartqa --split val --config configs/eval_chartqa.yaml
CMD = "cdt-train --stage smoke --config configs/model_qwen3vl2b.yaml"
subprocess.run(CMD, shell=True, cwd="/kaggle/working/repo", check=True)


In [ ]:
# Outputs are pushed to the private HF repo as they are produced, so a killed
# session loses nothing. This is only a convenience listing.
subprocess.run(["ls", "-R", "/kaggle/working/cdt-outputs"], check=False)
